# 삼성전자 가전 Agentic RAG 파이프라인 구성

이 노트북은 LangChain 1.0+ Tool Calling Agent 아키텍처를 기반으로 하며, 질문의 의도에 따라 TV, 냉장고, 에어컨 전용 리트리버를 선택적으로 사용하는 멀티 도구 에이전트를 구축합니다.

### Open AI API Key

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv("/home/user/MultiAgents/.env")

# 모델 정의
model = init_chat_model(
    "gpt-4o-mini",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # 핵심: OpenAI SDK의 base_url과 동일 개념
    temperature=0,    
)

### Agent 구성하기

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

### RAG tool을 Agent에 통합하기
* Chroma DB를 활용해 RAG 시스템을 구성합니다.
* 구성한 RAG 시스템을 tool로 제작합니다.
* 제작한 툴을 ReAct Agent로 구현합니다.

### Load Data
* 준비한 데이터를 활용해 각 Query Engine을 제작할 예정입니다.
* 각 Query Engine의 경우 서로 다른 데이터를 이용해 제작됩니다.
* 총 3개 종류의 Query Engine tool을 제작해봅시다.

In [ ]:
tv_path = "/home/user/MultiAgents/Day_1/Projects/data/삼성TV"
refri_path = "/home/user/MultiAgents/Day_1/Projects/data/삼성냉장고"
aircon_path = "/home/user/MultiAgents/Day_1/Projects/data/삼성에어컨"

### RAG 구성을 위한 Embedding 함수

In [ ]:
embed_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],   # ← init_chat_model과 동일
)

In [ ]:
# 함수 파라미터에 chunk_size와 chunk_overlap을 추가하고 기본값을 설정합니다.
def make_retriever(dir_path, collection_name, glob_pattern="**/*.txt",
                   chunk_size=1024, chunk_overlap=128, k=3):
    """지정된 디렉토리의 문서를 로드하여 리트리버 반환"""
    loader = DirectoryLoader(
        dir_path,  # 문서를 불러올 디렉토리 경로입니다.
        glob=glob_pattern,  # 어떤 파일들을 읽어올지 지정하는 패턴입니다.
        loader_cls=TextLoader,  # 각 파일을 텍스트 문서로 읽기 위한 로더 클래스를 사용합니다.
        loader_kwargs={"autodetect_encoding": True}, # 파일 인코딩을 자동 감지하도록 설정합니다.
        show_progress=True  # 문서 로딩 진행 상황을 출력합니다.
    )
    docs = loader.load()  # 디렉토리 내 문서들을 모두 로드합니다.

    # 문서 분할 (Chunking) - 전달받은 파라미터를 사용하도록 수정!
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,  # 한 청크에 들어갈 최대 문자 수를 설정합니다.
        chunk_overlap=chunk_overlap  # 인접 청크끼리 겹치게 할 문자 수를 설정합니다.
    )
    split_docs = splitter.split_documents(docs)  # 로드한 문서들을 여러 청크로 나눕니다.

    # 벡터 DB 구성
    vector = Chroma.from_documents(
        documents=split_docs,  # 분할된 문서 청크들을 벡터 DB에 저장합니다.
        embedding=embed_model,  # 각 청크를 임베딩하기 위한 임베딩 모델을 사용합니다.
        collection_name=collection_name  # Chroma 내부에서 사용할 컬렉션 이름을 지정합니다.
    )

    return vector.as_retriever(search_kwargs={"k": k})  # 상위 k개 문서를 검색하는 리트리버를 반환합니다.

In [ ]:
# Retriever 인스턴스화
retriever_tv_tool = make_retriever(tv_path, "samsung_tv")
retriever_fridge_tool = make_retriever(refri_path, "samsung_fridge")
retriever_ac_tool = make_retriever(aircon_path, "samsung_aircon")

In [ ]:
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 검색된 문서를 바탕으로 답변을 합성(Synthesis)하는 공통 함수
def synthesize_response(query: str, retriever) -> str:
    # 1. 리트리버를 통한 문서 검색
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])

    # 2. 정보 합성을 위한 프롬프트 템플릿 정의
    prompt = PromptTemplate.from_template(
        
        """당신은 삼성전자 가전 제품 상담 전문가입니다.
아래 검색된 문서 내용만 근거로 사용자의 질문에 답변하세요.
문서에 없는 내용은 추측하지 말고 확인되지 않는다고 말하세요.

[검색 문서]
{context}

[사용자 질문]
{query}

[답변]
"""

    )

    chain = prompt | model | StrOutputParser()
    return chain.invoke({"context": context, "query": query})


@tool
def search_samsung_tv(query: str) -> str:
    """Search Samsung TV documents and answer questions about Samsung TV products."""
    return synthesize_response(query, retriever_tv_tool)

@tool
def search_samsung_fridge(query: str) -> str:
    """Search Samsung refrigerator documents and answer questions about Samsung fridge products."""
    return synthesize_response(query, retriever_fridge_tool)

@tool
def search_samsung_ac(query: str) -> str:
    """Search Samsung air conditioner documents and answer questions about Samsung AC products."""
    return synthesize_response(query, retriever_ac_tool)

# 도구 리스트 등록
tools = [search_samsung_tv, search_samsung_fridge, search_samsung_ac]

### Agent 구성
* Agent가 실제 동작을 진행하면서 rool을 참고하도록 구성합니다.
* tool, functions 모두 입력 가능하며, system prompt를 잘 구성해 tool를 선택할 수 있도록 프롬프트를 입력합니다.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# 시스템 프롬프트 설정
system_instructions = """당신은 삼성전자 가전 제품 안내를 돕는 Agentic RAG 어시스턴트입니다.

사용자의 질문 의도를 파악해 반드시 적절한 검색 도구를 선택하세요.
- TV, Neo QLED, OLED, The Frame 등 TV 관련 질문: search_samsung_tv
- 냉장고, 비스포크 냉장고, 김치냉장고 등 냉장고 관련 질문: search_samsung_fridge
- 에어컨, 무풍 에어컨, 냉방/공기청정 등 에어컨 관련 질문: search_samsung_ac

질문이 여러 제품군을 비교하거나 함께 묻는 경우 필요한 도구를 모두 호출한 뒤 비교해서 답변하세요.
답변은 한국어로 작성하고, 검색 결과에 근거한 핵심 기능과 차별점을 간결하게 정리하세요.
검색 결과에 없는 내용은 추측하지 말고 문서에서 확인되지 않는다고 말하세요.
"""

# 에이전트 생성 (LangChain 1.0 표준)
# AgentExecutor나 복잡한 MessagesPlaceholder 템플릿 없이 직관적으로 생성 가능
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_instructions
)


### Invoke
* RAG 검색을 수행합니다.


In [ ]:
# ==========================================
# 4. 테스트 실행
# ==========================================

def print_agent_process(response):
    """에이전트의 메시지 기록을 분석하여 도구 호출 내역과 최종 답변을 출력하는 헬퍼 함수"""
    print("\n[에이전트 동작 과정]")  # 에이전트가 어떤 순서로 동작했는지 출력 시작을 알립니다.
    for msg in response["messages"]:  # 에이전트가 남긴 모든 메시지를 순차적으로 확인합니다.
        # 1. AI가 도구를 호출한 경우 (Tool Call)
        if hasattr(msg, 'tool_calls') and msg.tool_calls:  # 현재 메시지에 도구 호출 정보가 있는지 확인합니다.
            for tool_call in msg.tool_calls:  # 한 메시지 안의 여러 도구 호출을 하나씩 순회합니다.
                print(f"🛠️ [도구 선택]: {tool_call['name']}")  # 선택된 도구 이름을 출력합니다.
                print(f"🔍 [검색 질의]: {tool_call['args']}")  # 도구에 전달된 인자(검색 질의)를 출력합니다.

        # 2. 도구 실행이 완료되어 결과를 반환한 경우 (Tool Message)
        elif getattr(msg, 'type', '') == 'tool':  # 현재 메시지가 도구 실행 결과인지 확인합니다.
            # 내용이 길 수 있으므로 앞 100자만 잘라서 출력합니다.
            content_snippet = msg.content[:100].replace('\n', ' ')  # 검색 결과가 너무 길면 앞 100자만 잘라서 한 줄로 정리합니다.
            print(f"📄 [검색 결과]: {content_snippet}...\n")  # 도구가 반환한 검색 결과 일부를 출력합니다.

    # 3. 최종 답변 출력
    print("-" * 20)  # 최종 답변과 중간 과정을 구분하는 구분선을 출력합니다.
    print(f"🎯 [최종 답변]\n{response['messages'][-1].content}\n")  # 마지막 메시지를 최종 답변으로 출력합니다.


print("=== 질문 1: 단일 카테고리 ===")  # 첫 번째 테스트 케이스 제목을 출력합니다.
response1 = agent.invoke({
    "messages": [{"role": "user", "content": "요즘 인기있는 삼성 Neo QLED TV의 주요 기능에 대해 알려줘."}] 
})
print_agent_process(response1)  # 첫 번째 질문에 대한 에이전트의 동작 과정과 최종 답변을 출력합니다.

print("=" * 40)  # 두 테스트 케이스를 구분하는 구분선을 출력합니다.

print("=== 질문 2: 복합 카테고리 비교 ===")  # 두 번째 테스트 케이스 제목을 출력합니다.
response2 = agent.invoke({
    "messages": [{"role": "user", "content": "비스포크 냉장고와 무풍 에어컨의 차별점을 각각 비교해서 설명해줄래?"}]
})
print_agent_process(response2)  # 두 번째 질문에 대한 에이전트의 동작 과정과 최종 답변을 출력합니다.